In [ ]:
%load_ext autoreload
%autoreload 2


import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from pendulibrary.integrate import integrate_state
from pendulibrary.continuation import adaptive_cont
from pendulibrary.common_targetters import single_fixed
from pendulibrary.utils import get_x0_corrected

int_tol = 1e-14

In [ ]:
Lr = 1.0
Mr = 1.0

# x0_ud, T_ud, tan_ud = get_x0_corrected(np.pi, 0.0, Lr, Mr, 5e-5)
# x0_du, T_du, tan_du = get_x0_corrected(0.0, np.pi, Lr, Mr, 5e-5)
x0s_dd, Ts_dd, tans_dd = get_x0_corrected(0.0, 0.0, Lr, Mr, 5e-5)

In [ ]:
cont_kwargs = dict(
    s0=5e-5, s_min=5e-5, S=20.0, tol=1e-8, max_iter=8, target_iter=4, rate=1.15
)

dd_ind = 1
targ = single_fixed(0, x0s_dd[dd_ind][0], 2, Lr, Mr, int_tol)
func = targ.g_dg_stm
X0, tan = targ.get_X(x0s_dd[dd_ind], Ts_dd[dd_ind]), tans_dd[dd_ind]

if np.dot(X0[1:-1], tan[1:-1]) < 0:
    tan *= -1
Xs, eig_vals, (DFs, tangents, stms) = adaptive_cont(
    X0, func, tan, **cont_kwargs, exact_tangent=True
)

In [ ]:
cont_kwargs2 = dict(
    s0=5e-5, s_min=1e-5, S=20.0, tol=1e-6, max_iter=8, target_iter=4, rate=1.15
)
targ = single_fixed(0, x0s_dd[dd_ind][0], 2, Lr, Mr, int_tol)
func = targ.g_dg_stm

Xs2, eig_vals2, (DFs2, tangents2, stms2) = adaptive_cont(
    Xs[-1], func, tangents[-1], **cont_kwargs, exact_tangent=True
)

In [ ]:
x0s = np.array([targ.get_x0(X) for X in Xs])
periods = np.array([targ.get_period(X) for X in Xs])
np.savez(
    "../database/DD_short_period_initial",
    x0s=x0s,
    periods=periods,
    eigs=eig_vals,
    tangents=tangents,
    params=np.array([Lr, Mr]),
)

In [ ]:
from pendulibrary.plotters import plot_timeline

ts, xs, fs = integrate_state(
    targ.get_x0(Xs[30]), targ.get_period(Xs[30]), Lr, Mr, 1e-14
)

fig = plot_timeline(xs, ts, fs, Lr, 25)
plt.show()

In [ ]:
#xs[:, -1] - xs[:, 0]
#eig_vals.shape
#eig_vals[:,:2]
#eig_vals[-1]

nontrivial_eigs = [row[np.argsort(np.abs(row-1))[:2]] for row in eig_vals]
print(nontrivial_eigs)

In [ ]:
eig_vals[-1]

In [ ]:
from pendulibrary.targeter import dc_underconstrained

X, dF, stm_full = dc_underconstrained(Xs[-1], func, 1e-12, debug=True, max_iter=20)

In [ ]:
_,_,stm = targ.g_dg_stm(X)
np.linalg.eigvals(stm)